# Mora plotting notebook

This notebook consolidates the plotting workflow for the Mora solver outputs.
It reads the standard text files written by the executable and reproduces the main profile, field, spectrum, and history plots in one place.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_solver_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    header = None
    for line in path.read_text().splitlines():
        if line.startswith('#'):
            header = line[1:].strip().split()
            break

    if header is None:
        raise ValueError(f'No header line found in {path}')

    data_lines = [line for line in path.read_text().splitlines() if line.strip() and not line.lstrip().startswith('#')]
    if not data_lines:
        return pd.DataFrame(columns=header)

    cleaned = '\n'.join(line.replace('*', ' NaN ') for line in data_lines)
    from io import StringIO
    frame = pd.read_csv(StringIO(cleaned), sep=r',|\s+', header=None, engine='python', na_values=['NaN', 'nan', 'Infinity', '-Infinity'])
    ncols = min(len(header), frame.shape[1])
    frame = frame.iloc[:, :ncols]
    frame.columns = header[:ncols]
    frame = frame.apply(pd.to_numeric, errors='coerce')
    return frame


repo_root = find_repo_root()
default_data_dir = repo_root / 'build'
data_dir = default_data_dir if default_data_dir.exists() else repo_root
viz_dir = repo_root / 'scripts' / 'viz'
viz_dir.mkdir(parents=True, exist_ok=True)

print(f'repo_root = {repo_root}')
print(f'data_dir   = {data_dir}')
print(f'viz_dir    = {viz_dir}')

repo_root = /Users/42d/Mora
data_dir   = /Users/42d/Mora/build
viz_dir    = /Users/42d/Mora/scripts/viz


In [3]:
available = {
    'profil': data_dir / 'profil.txt',
    'spectres': data_dir / 'spectres.txt',
    'conservation': data_dir / 'conservation.txt',
    'historique': data_dir / 'historique.txt',
}

for name, path in available.items():
    print(f'{name:12s} -> {path} ({"present" if path.exists() else "missing"})')

profil       -> /Users/42d/Mora/build/profil.txt (present)
spectres     -> /Users/42d/Mora/build/spectres.txt (present)
conservation -> /Users/42d/Mora/build/conservation.txt (present)
historique   -> /Users/42d/Mora/build/historique.txt (present)


In [4]:
profiles = read_solver_table(available['profil']) if available['profil'].exists() else None
spectra = read_solver_table(available['spectres']) if available['spectres'].exists() else None
conservation = read_solver_table(available['conservation']) if available['conservation'].exists() else None
history = read_solver_table(available['historique']) if available['historique'].exists() else None

for name, frame in [('profiles', profiles), ('spectra', spectra), ('conservation', conservation), ('history', history)]:
    if frame is None:
        print(f'{name}: unavailable')
    else:
        print(f'{name}: shape={frame.shape}')
        display(frame.head())

/var/folders/99/pbwmmd_11vq4grmtfv52lnxm0000gp/T/ipykernel_51662/3945217194.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  frame = pd.read_csv(path, comment='#', delim_whitespace=True, header=None)
/var/folders/99/pbwmmd_11vq4grmtfv52lnxm0000gp/T/ipykernel_51662/3945217194.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  frame = pd.read_csv(path, comment='#', delim_whitespace=True, header=None)


EmptyDataError: No columns to parse from file

In [ ]:
if profiles is None or spectra is None:
    raise RuntimeError('profil.txt and spectres.txt are required for the main spatial and spectral plots.')

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

ax = axes[0, 0]
ax.plot(profiles['xt'], profiles['ni'], label='ni', lw=2)
ax.plot(profiles['xt'], profiles['ne'], label='ne', lw=2)
if 'nhot' in profiles.columns:
    ax.plot(profiles['xt'], profiles['nhot'], label='nhot', lw=2, ls='--')
ax.set_title('Density profiles')
ax.set_xlabel('x')
ax.set_ylabel('density')
ax.legend()

ax = axes[0, 1]
ax.plot(profiles['xt'], profiles['E'], color='tab:red', lw=2)
ax.set_title('Electric field')
ax.set_xlabel('x')
ax.set_ylabel('E')

ax = axes[1, 0]
ax.plot(profiles['xt'], profiles['rho'], color='tab:green', lw=2)
ax.axhline(0.0, color='black', lw=1, alpha=0.5)
ax.set_title('Charge separation')
ax.set_xlabel('x')
ax.set_ylabel('rho')

ax = axes[1, 1]
ax.plot(spectra['Emoy'], spectra['dndE'], color='tab:purple', lw=2)
ax.set_title('Energy spectrum')
ax.set_xlabel('Energy')
ax.set_ylabel('dN/dE')
ax.set_yscale('log')

fig.suptitle('Mora solver diagnostics', fontsize=16)
plt.show()

In [ ]:
if conservation is not None or history is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    if conservation is not None:
        axes[0, 0].plot(conservation['time'], conservation['En_ion'], label='En_ion', lw=2)
        axes[0, 0].plot(conservation['time'], conservation['En_elec'], label='En_elec', lw=2)
        if 'Th' in conservation.columns:
            axes[0, 1].plot(conservation['time'], conservation['Th'], color='tab:orange', lw=2)
            axes[0, 1].set_title('Hot-electron temperature')
            axes[0, 1].set_xlabel('time')
            axes[0, 1].set_ylabel('Th')
        axes[0, 0].set_title('Energy budget')
        axes[0, 0].set_xlabel('time')
        axes[0, 0].set_ylabel('energy')
        axes[0, 0].legend()

    if history is not None:
        axes[1, 0].plot(history['time'], history['v(ivmax)'], color='tab:red', lw=2)
        axes[1, 0].set_title('Front velocity history')
        axes[1, 0].set_xlabel('time')
        axes[1, 0].set_ylabel('v(ivmax)')

        if 'E(ivmax)' in history.columns:
            axes[1, 1].plot(history['time'], history['E(ivmax)'], color='tab:blue', lw=2)
            axes[1, 1].set_title('Front field history')
            axes[1, 1].set_xlabel('time')
            axes[1, 1].set_ylabel('E(ivmax)')

    for ax in axes.flat:
        if not ax.has_data():
            ax.axis('off')

    plt.show()
else:
    print('No time-history files are available yet.')

In [ ]:
save_figures = False

if save_figures and profiles is not None and spectra is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axes[0, 0].plot(profiles['xt'], profiles['ni'], label='ni', lw=2)
    axes[0, 0].plot(profiles['xt'], profiles['ne'], label='ne', lw=2)
    if 'nhot' in profiles.columns:
        axes[0, 0].plot(profiles['xt'], profiles['nhot'], label='nhot', lw=2, ls='--')
    axes[0, 0].legend()
    axes[0, 0].set_title('Density profiles')

    axes[0, 1].plot(profiles['xt'], profiles['E'], color='tab:red', lw=2)
    axes[0, 1].set_title('Electric field')

    axes[1, 0].plot(profiles['xt'], profiles['rho'], color='tab:green', lw=2)
    axes[1, 0].set_title('Charge separation')

    axes[1, 1].plot(spectra['Emoy'], spectra['dndE'], color='tab:purple', lw=2)
    axes[1, 1].set_yscale('log')
    axes[1, 1].set_title('Energy spectrum')

    output_path = viz_dir / 'mora_combined_diagnostics.png'
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved {output_path}')